In [ ]:
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import monotonically_increasing_id
import pyspark.pandas as ps

import os

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


In [2]:
from hypex.matching import Matching
from hypex.ml.faiss import FaissNearestNeighbors
from hypex.transformers import TypeCaster
from hypex.dataset import Dataset, InfoRole, TreatmentRole, FeatureRole, TargetRole, ExperimentData, AdditionalMatchingRole, AdditionalStatisticRole, DisabledRole
from hypex.utils import BackendsEnum
from hypex.experiments import Experiment, OnRoleExperiment
from hypex.comparators import MahalanobisDistance
from hypex.encoders.encoders import DummyEncoder
from hypex.comparators import TTest, Chi2Test
from hypex.comparators.distances import MahalanobisDistance
from hypex.operators import Bias, MatchingMetrics
from hypex.analyzers import MatchingAnalyzer

In [3]:
# --- 1. Настройки окружения для macOS (Важно!) ---
# На macOS иногда возникают проблемы с форком процессов Java (Executor'ы не стартуют).
# Эта переменная часто решает проблему "Connection refused" или краши при запуске local-cluster
os.environ['OBJC_DISABLE_INITIALIZE_FORK_SAFETY'] = 'YES'

# Очистка старых сессий и переменных (как у вас было)
try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        existing_spark.stop()
        print("✅ Существующая сессия остановлена.")
except:
    pass

for key in list(os.environ.keys()):
    if 'SPARK' in key or 'JAVA_OPTS' in key:
        del os.environ[key]

# --- 2. Конфигурация Кластера ---
# Формат: local-cluster[число_воркеров, ядер_на_воркер, память_на_воркер_в_МБ]
# Мы просим 2 экзекутора, по 1 ядру, по 2 ГБ памяти каждый
NUM_EXECUTORS = 2
CORES_PER_EXECUTOR = 4
MEMORY_PER_EXECUTOR_MB = 2048 

MASTER_URL = f"local-cluster[{NUM_EXECUTORS}, {CORES_PER_EXECUTOR}, {MEMORY_PER_EXECUTOR_MB}]"

print(f"🚀 Запуск в режиме: {MASTER_URL}")

sp_s = (SparkSession.builder
    .master(MASTER_URL)
    .appName("LocalClusterTest")
    # Память драйвера (остается у вас)
    .config("spark.driver.memory", "2g") 
    # Память экзекутора (должна соответствовать или быть меньше чем в master URL)
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "4")
    .config("spark.executor.instances", NUM_EXECUTORS)
    # Увеличиваем память под оверхед, чтобы избежать ошибок выделения памяти
    .config("spark.memory.fraction", "0.6")
    .config("spark.sql.shuffle.partitions", "4") # Для тестов меньше дефолтных 200
    .getOrCreate()
)

sp_s.sparkContext.setLogLevel("WARN")

# --- 3. Проверка конфигурации ---
print(f"✅ Сессия создана.")
print(f"Driver Memory Config: {sp_s.conf.get('spark.driver.memory')}")
print(f"Executor Memory Config: {sp_s.conf.get('spark.executor.memory')}")

# Проверка количества экзекуторов (может занять пару секунд на старт)
import time
time.sleep(3) 
num_executors = len(sp_s.sparkContext.parallelize(range(10), NUM_EXECUTORS).glom().collect())
print(f"📊 Активных экзекуторов (проверка через RDD): {num_executors}")

# --- 4. Тест на распределение (Пример) ---
# Чтобы убедиться, что задача ушла на экзекуторы, а не осталась на драйвере
def print_executor_info(iterator):
    import os
    # Получаем ID экзекутора из переменных окружения процесса
    executor_id = os.environ.get('SPARK_EXECUTOR_ID', 'Driver/Local')
    process_id = os.getpid()
    return [f"Executor ID: {executor_id}, PID: {process_id}"]

# Создаем датафрейм и применяем трансформацию
df = sp_s.range(0, 10, 1, 4) # 4 партиции
result = df.rdd.mapPartitions(print_executor_info).collect()

print("\n🖥️ Где выполнялись задачи:")
for line in result:
    print(line)

# Не забывайте останавливать сессию в конце скрипта, так как процессы тяжелые
# sp_s.stop() 

🚀 Запуск в режиме: local-cluster[2, 4, 2048]


26/06/24 11:37:03 WARN Utils: Your hostname, eric-Katana-17-B12UCR resolves to a loopback address: 127.0.1.1; using 10.240.68.114 instead (on interface wlo1)
26/06/24 11:37:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/24 11:37:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Сессия создана.
Driver Memory Config: 2g
Executor Memory Config: 2g


📊 Активных экзекуторов (проверка через RDD): 2



🖥️ Где выполнялись задачи:
Executor ID: Driver/Local, PID: 560040
Executor ID: Driver/Local, PID: 560046
Executor ID: Driver/Local, PID: 560092
Executor ID: Driver/Local, PID: 560099


In [4]:
n = 5000  # увеличьте для теста IVF-индексов
df = pd.DataFrame({
    "treatment": np.random.choice([0, 1], size=n, p=[0.6, 0.4]),
    "feat_num_1": np.random.normal(loc=10, scale=3, size=n),
    "feat_num_2": np.random.normal(loc=-2, scale=1.5, size=n),
    "feat_cat": np.random.choice(["A", "B", "C"], size=n),
    "target": np.random.normal(loc=100, scale=10, size=n)
})

In [2]:
nn = 200
index_df = pd.DataFrame(
    {
        '0': np.random.randint(0, 5000, nn),
        '1': np.random.randint(0, 5000, nn),
        '2': np.random.randint(0, 5000, nn),
        '3': np.random.randint(0, 5000, nn),
        '4': np.random.randint(0, 5000, nn),
        'group': [0] * (nn//4) + [1] * (nn//4) + [2] * (nn//4) + [3] * (nn//4)
    }
)

# index_df = pd.concat([index_df, pd.DataFrame(data={
#     '0': np.nan,
#     '1': np.nan,
#     '2': np.nan,
#     '3': np.nan,
#     '4': np.nan,
#     'group': np.nan
# }, index=[0])]).reset_index(drop=True)
# index_df

In [ ]:
session = (
            SparkSession.builder
            .master("local[*]")
            .config("spark.driver.memory", "4g")
            .config("spark.executor.memory", "4g")
            .config("spark.memory.fraction", "0.8") 
            .config("spark.memory.storageFraction", "0.3")
            # .config("spark.jars.packages", "ch.cern.sparkmeasure:spark-measure_2.12:0.23") 
            .getOrCreate()
          )

In [ ]:
index_ds = Dataset(
    roles={
        'group': TargetRole()
    },
    # data=index_df
    data=session.createDataFrame(index_df),
    # data=sp_s.createDataFrame(index_df),
    session=session
    # session=sp_s
)

index_ds

In [ ]:
"""
PANDAS case
"""
# 4. Обёртка в Dataset фреймворка
roles = {
    "treatment": TreatmentRole(),
    "feat_num_1": FeatureRole(),
    "feat_num_2": FeatureRole(),
    "feat_cat": FeatureRole(str),
    "target": TargetRole(),
    # "index": FeatureRole()  # индекс тоже должен быть в ролях, чтобы не отфильтровался
}

dataset = Dataset(
    roles=roles,
    data=df,
    backend=BackendsEnum.pandas,
)

pandas_experiment = Experiment(
    executors=[
        DummyEncoder(),
        MahalanobisDistance(
            grouping_role=TreatmentRole(),
            weights=None
        ),
        TypeCaster(
            dtype={int: float},
            roles=[FeatureRole(), TargetRole()],
        ),
        FaissNearestNeighbors(
            grouping_role=TreatmentRole(),
            two_sides=True,
            test_pairs=False,
            faiss_mode="auto",
            n_neighbors=2,
        ),
        Bias(
            grouping_role=TreatmentRole(), 
            target_roles=[TargetRole()]
        ),
        MatchingMetrics(
                grouping_role=TreatmentRole(),
                target_roles=[TargetRole()],
                metric="ate",
                n_neighbors=2,
        ),
        MatchingAnalyzer(),
        OnRoleExperiment(
            executors=[
                TTest(
                    grouping_role=TreatmentRole(),
                    compare_by="matched_pairs",
                    baseline_role=AdditionalMatchingRole(),
                ),
                Chi2Test(
                    grouping_role=TreatmentRole(),
                    compare_by="matched_pairs",
                    baseline_role=AdditionalMatchingRole()
                )
            ],
            role=FeatureRole()
        )
    ]
)
pandas_result = pandas_experiment.execute(ExperimentData(dataset))

DummyEncoder
executor.key = DummyEncoder┴┴; dt = 0.0147c
MahalanobisDistance
executor.key = MahalanobisDistance┴┴['feat_num_1', 'feat_num_2', 'DummyEncoder||_feat_cat_B', 'DummyEncoder||_feat_cat_C']; dt = 0.0205c
TypeCaster
executor.key = TypeCaster┴┴; dt = 0.0072c
FaissNearestNeighbors
executor.key = FaissNearestNeighbors┴┴; dt = 0.4529c
Bias
executor.key = Bias┴┴['target', 'target_matched']; dt = 0.1089c
MatchingMetrics
executor.key = MatchingMetrics┴┴['target', 'target_matched']; dt = 0.2740c
MatchingAnalyzer
executor.key = MatchingAnalyzer┴┴; dt = 0.0039c
OnRoleExperiment
GroupTTest
executor.key = TTest┴┴; dt = 0.0804c
GroupTTest
executor.key = TTest┴┴; dt = 0.0751c
GroupTTest


/home/eric/HypEx/HypEx/hypex/comparators/abstract.py:242: UserWarning: baseline_field_data must have only one column when the comparison is done by matched_pairs. 2 passed. FaissNearestNeighbors┴┴┴0 will be used.
  warnings.warn(
/home/eric/HypEx/HypEx/hypex/comparators/abstract.py:242: UserWarning: baseline_field_data must have only one column when the comparison is done by matched_pairs. 2 passed. FaissNearestNeighbors┴┴┴0 will be used.
  warnings.warn(
/home/eric/HypEx/HypEx/hypex/comparators/abstract.py:242: UserWarning: baseline_field_data must have only one column when the comparison is done by matched_pairs. 2 passed. FaissNearestNeighbors┴┴┴0 will be used.
  warnings.warn(


executor.key = TTest┴┴; dt = 0.0754c
GroupTTest
executor.key = TTest┴┴; dt = 0.0784c
executor.key = OnRoleExperiment┴┴; dt = 0.3319c


/home/eric/HypEx/HypEx/hypex/comparators/abstract.py:242: UserWarning: baseline_field_data must have only one column when the comparison is done by matched_pairs. 2 passed. FaissNearestNeighbors┴┴┴0 will be used.
  warnings.warn(


In [5]:
pandas_result.analysis_tables

{'MatchingAnalyzer┴┴':      Effect Size  Standard Error   P-value  CI Lower  CI Upper
 ATT     0.094620        0.286763  0.741431 -0.467435  0.656675
 ATC     0.327411        0.590379  0.579183 -0.829731  1.484553
 ATE     0.232665        0.384621  0.545232 -0.521192  0.986522
 
 3 rows × 5 columns,
 'GroupTTest┴┴feat_num_1':                   p-value  statistic  pass
 (0,)┆feat_num_1  0.998199  -0.002258   0.0
 (1,)┆feat_num_1  0.991377  -0.010808   0.0
 
 2 rows × 3 columns,
 'GroupTTest┴┴feat_num_2':                   p-value  statistic  pass
 (0,)┆feat_num_2  0.966370   0.042163   0.0
 (1,)┆feat_num_2  0.949143   0.063787   0.0
 
 2 rows × 3 columns,
 'GroupTTest┴┴DummyEncoder||_feat_cat_B':                                  p-value  statistic  pass
 (0,)┆DummyEncoder┴┴_feat_cat_B  0.978110  -0.027439   0.0
 (1,)┆DummyEncoder┴┴_feat_cat_B  0.921288   0.098818   0.0
 
 2 rows × 3 columns,
 'GroupTTest┴┴DummyEncoder||_feat_cat_C':                                 p-value  statistic  pa

In [ ]:
pandas_result.analysis_tables

In [5]:
"""
PYSPARK case
"""
# 3. Конвертация в Spark + ОБЯЗАТЕЛЬНАЯ колонка `index` (требование Faiss)
spark_df = sp_s.createDataFrame(df)

# 4. Обёртка в Dataset фреймворка
roles = {
    "treatment": TreatmentRole(),
    "feat_num_1": FeatureRole(),
    "feat_num_2": FeatureRole(),
    "feat_cat": FeatureRole(str),
    "target": TargetRole(),
    # "index": FeatureRole()  # индекс тоже должен быть в ролях, чтобы не отфильтровался
}

dataset = Dataset(
    roles=roles,
    data=spark_df,
    # data=session.createDataFrame(df),
    # data=df,
    backend=BackendsEnum.spark,
    session=sp_s,
    # session=session
)

spark_experiment = Experiment(
    executors=[
        DummyEncoder(),
        MahalanobisDistance(
            grouping_role=TreatmentRole(),
            weights=None
        ),
        TypeCaster(
            dtype={int: float},
            roles=[FeatureRole(), TargetRole()],
        ),
        FaissNearestNeighbors(
            grouping_role=TreatmentRole(),
            two_sides=True,
            test_pairs=False,
            faiss_mode="auto",
            n_neighbors=2,
        ),
        Bias(
            grouping_role=TreatmentRole(), 
            target_roles=[TargetRole()]
        ),
        MatchingMetrics(
                grouping_role=TreatmentRole(),
                target_roles=[TargetRole()],
                metric="ate",
                n_neighbors=2,
        ),
        MatchingAnalyzer(),
        OnRoleExperiment(
            executors=[
                TTest(
                    grouping_role=TreatmentRole(),
                    compare_by="matched_pairs",
                    baseline_role=AdditionalMatchingRole(),
                ),
                Chi2Test(
                    grouping_role=TreatmentRole(),
                    compare_by="matched_pairs",
                    baseline_role=AdditionalMatchingRole()
                )
            ],
            role=FeatureRole()
        )
    ]
)
spark_result = spark_experiment.execute(ExperimentData(dataset))

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
26/06/24 11:37:21 WARN AttachDistributedSequenceExec: clean up cached RDD(21) in AttachDistributedSequenceExec(142)
26/06/24 11:37:22 WARN AttachDistributedSequenceExec: clean up

In [6]:
spark_result.analysis_tables

{'MatchingAnalyzer┴┴':      Effect Size  Standard Error   P-value  CI Lower  CI Upper
 ATT    -0.486245        0.267961  0.069584 -1.011449  0.038959
 ATC    -0.420587        0.568886  0.459714 -1.535603  0.694429
 ATE    -0.447047        0.365801  0.221668 -1.164017  0.269923
 
 3 rows × 5 columns,
 'StatsTTest┴┴feat_num_1┆stats':    mean┆feat_num_1  std┆feat_num_1  count┆feat_num_1  mean┆feat_num_1_matched  std┆feat_num_1_matched  count┆feat_num_1_matched
 1         9.922975        3.048191            2015.0                 9.922305                3.033455                    2015.0
 0        10.030395        3.018181            2985.0                10.030150                2.991954                    2985.0
 
 2 rows × 6 columns,
 'StatsTTest┴┴feat_num_1':                p-value  statistic  pass
 1┆feat_num_1  0.994419   0.006996   0.0
 0┆feat_num_1  0.997484   0.003153   0.0
 
 2 rows × 3 columns,
 'StatsTTest┴┴feat_num_2┆stats':    mean┆feat_num_2  std┆feat_num_2  count┆feat_num_2

In [7]:
spark_result.ds.unpersist()

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning:

,treatment,feat_num_1,feat_num_2,feat_cat,target,DummyEncoder┴┴_feat_cat_B,DummyEncoder┴┴_feat_cat_C,FaissNearestNeighbors┴┴┴0,FaissNearestNeighbors┴┴┴1,"Bias┴┴['target', 'target_matched']",MatchingMetrics┴┴
2,1,5.06023,-0.556955,A,84.504535,0.0,0.0,3863,2547,-0.003931,97.783749
4,0,13.95785,-2.741227,C,106.990887,0.0,1.0,1109,3332,-0.067205,94.537581
5,0,9.146479,-3.311966,A,96.234828,0.0,0.0,4964,3613,0.025635,106.653399
8,1,5.034941,-3.059856,A,81.477265,0.0,0.0,4695,2004,-0.00085,106.640616
12,1,5.722654,-1.758047,A,89.54213,0.0,0.0,129,3788,-0.006718,103.792952
...,...,...,...,...,...,...,...,...,...,...,...
4995,0,14.442877,-3.245186,A,88.12091,0.0,0.0,2985,578,-0.027309,100.867227
4996,1,4.339891,-0.885904,A,99.062302,0.0,0.0,4569,1593,-0.003714,98.765986
4997,0,7.819863,-2.15934,B,110.228421,1.0,0.0,2124,2906,-0.034089,100.82319
4998,0,16.152397,-2.458476,A,121.83119,0.0,0.0,4348,2068,-0.018136,93.890973


In [12]:
spark_result.variables["MahalanobisDistance┴┴['feat_num_1', 'feat_num_2', 'DummyEncoder||_feat_cat_B', 'DummyEncoder||_feat_cat_C']"]["['feat_num_1', 'feat_num_2', 'DummyEncoder┴┴_feat_cat_B', 'DummyEncoder┴┴_feat_cat_C']"]

,feat_num_1,feat_num_2,DummyEncoder┴┴_feat_cat_B,DummyEncoder┴┴_feat_cat_C
feat_num_1,0.329423,0.005064,-0.002166,-0.001165
feat_num_2,0.000000,0.674589,0.004112,0.004982
DummyEncoder┴┴_feat_cat_B,0.000000,0.000000,2.103345,1.198875
DummyEncoder┴┴_feat_cat_C,0.000000,0.000000,0.000000,2.456418


In [15]:
spark_result.field_search(AdditionalStatisticRole())

[]

In [6]:
spark_result.ds

26/06/22 16:04:20 WARN AttachDistributedSequenceExec: clean up cached RDD(2762) in AttachDistributedSequenceExec(67867)
26/06/22 16:04:20 WARN AttachDistributedSequenceExec: clean up cached RDD(2768) in AttachDistributedSequenceExec(67929)
26/06/22 16:04:21 WARN AttachDistributedSequenceExec: clean up cached RDD(2774) in AttachDistributedSequenceExec(67949)
26/06/22 16:04:21 WARN AttachDistributedSequenceExec: clean up cached RDD(2780) in AttachDistributedSequenceExec(68063)
26/06/22 16:04:21 WARN AttachDistributedSequenceExec: clean up cached RDD(2786) in AttachDistributedSequenceExec(68136)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
26/06/22 16:04:25 WARN AttachDistributedSequenceExec: clean up cached RDD(3069) in AttachDistributedSe

,treatment,feat_num_1,feat_num_2,feat_cat,target,DummyEncoder┴┴_feat_cat_B,DummyEncoder┴┴_feat_cat_C,FaissNearestNeighbors┴┴┴0,FaissNearestNeighbors┴┴┴1,"Bias┴┴['target', 'target_matched']"
631,0,10.518327,1.658239,C,114.512744,0.0,1.0,3781,2320,-0.02296
633,1,13.715355,-1.307652,C,88.759142,0.0,1.0,4422,891,0.015414
634,0,9.189822,-1.588253,C,88.861215,0.0,1.0,2492,112,-0.006866
635,1,7.364567,-2.519785,A,123.255314,0.0,0.0,2121,1971,0.000071
650,1,9.10831,-2.103024,B,82.677929,1.0,0.0,3796,920,0.003096
...,...,...,...,...,...,...,...,...,...,...
4995,0,11.223812,-4.354177,A,94.573657,0.0,0.0,3718,645,-0.005981
4996,0,11.134184,-1.408844,C,96.364358,0.0,1.0,4635,4205,0.000283
4997,0,13.456249,-2.490945,B,83.544663,1.0,0.0,179,1530,0.012741
4998,0,6.167694,-5.471622,B,79.052354,1.0,0.0,2025,3832,0.02688


In [ ]:
# for label, ds in result.variables['Bias┴┴[\'target\', \'target_matched\']'].items():
#     ds.to_small_dataset().data.to_csv(f"{label}.csv")

In [6]:
spark_result.analysis_tables

{'MatchingAnalyzer┴┴':      Effect Size  Standard Error  P-value  CI Lower  CI Upper
 ATT    -0.386312             0.0      0.0 -0.386312 -0.386312
 ATC     0.223290             0.0      0.0  0.223290  0.223290
 ATE    -0.024031             0.0      0.0 -0.024031 -0.024031
 
 3 rows × 5 columns}

In [8]:
sp_s.stop()

In [ ]:
r = pd.read_csv('result_[5].csv', names=['index', 'value'])
r.dropna(subset=['index']).fillna(0)

In [ ]:

# 5. Настройка Matching
matching = Matching(
    distance="mahalanobis",
    # metric="ate",
    bias_estimation=False,      # отключаем для упрощения дебага
    quality_tests=["t-test"],   # только t-test для скорости
    faiss_mode="base",          # "base" → IndexFlatL2 (без IVF), проще отлаживать
    n_neighbors=1,
    encode_categories=True      # DummyEncoder включится автоматически
)

# 6. Запуск
print("🚀 Запуск пайплайна Matching...")
result_data = matching.execute(dataset)

print("✅ Выполнено успешно!")
print(f"📊 Additional Fields: {result_data.additional_fields.columns}")
print(f"📦 Groups Keys: {list(result_data.groups.keys())}")

In [ ]:
sp_s.stop()